# URP4-1 HQ v0.1

이 노트북은 검증된 generator/import/descriptor 모듈을 호출하는 관제판입니다. **사용자는 다음 Cell 1만 수정**합니다. 설정은 Cell 2에서 검증·파생·동결되며 이후 셀은 동결본을 우회할 수 없습니다.

- generated STL → native controlled route
- imported STL → IMSTL-005 oriented non-zero winding route
- original STP → strict/reference preflight route
- STL→STEP 자동 fallback 없음
- feature selection / training 기본 OFF 및 fail-closed


In [ ]:
# CELL 1 — MASTER CONTROLLER (사용자는 이 셀만 수정)
from pathlib import Path
from urp4.hq.v0_1.controller import (
    URP4Controller, WorkflowControl, PathControl, GeometryControl,
    ImportControl, GeneratorControl, LatticeControl, TPMSControl, VoxelControl,
    SliceControl, DescriptorControl, ArtifactControl, TrainingControl,
)

ROOT = Path.cwd().resolve()
if not (ROOT / 'urp4').is_dir():
    raise RuntimeError('URP4-1_DELIVERABLE 폴더를 작업 디렉터리로 열어주세요.')

CONTROLLER = URP4Controller(
    workflow=WorkflowControl(
        import_geometry=True,
        generate_geometry=False,
        extract_descriptors=True,
        feature_selection=False,
        training=False, run_batch=False,
    ),
    paths=PathControl(
        input_geometry=r'',  # 단일 실행 STL/STP 절대경로
        input_directory=r'',  # batch용; v0.1 실행은 잠김
        output_root='outputs', experiment_name='URP4-HQ',
        target_y='',
    ),
    import_control=ImportControl(
        prefer_original_stp=True, allow_imported_stl=True, reject_topology_risk=False,
    ),
    geometry=GeometryControl(
        model_id='MODEL-001',
        source_type='imported_stl',  # generated_stl | imported_stl | original_stp
        geometry_revision='user-input',
        expected_size_mm=40.0,
        normalize_imported_stl=True,
    ),
    generator=GeneratorControl(
        family='lattice_type_a',  # lattice_type_a | lattice_type_b | tpms | voxel
        random_seed=42, target_vf=0.30, requested_format='stl',  # TPMS는 현재 0.45~0.55만 허용
    ),
    lattice=LatticeControl(
        total_length_mm=40.0, cell_definition_mode='cells_per_axis',
        cells_per_axis=5, cell_size_mm=None,
    ),
    tpms=TPMSControl(registry_index=0, grid_n=40),
    voxel=VoxelControl(registry_index=0, grid_n=40, rows_per_mode=1),
    slicing=SliceControl(
        axis='z', physical_size_mm=40.0,
        definition_mode='derive_count_from_spacing',
        slice_count=None, slice_spacing_mm=0.05, pixel_resolution=1000,
        connectivity=8, min_component_pixels=2, threshold_rule='binary_nonzero_png_readback',
    ),
    descriptor=DescriptorControl(
        point_enabled=False, surface_enabled=False, slice_enabled=True,
        lattice_enabled=False, candidate_enabled=False,
    ),
    artifacts=ArtifactControl(
        image_policy='STREAMING_TEMP_PNG', overwrite=False,
        export_component_tables=True, visual_review_export=False,
    ),
    training=TrainingControl(),
)
CONTROLLER


In [ ]:
# CELL 2 — validate → derive → freeze
from urp4.hq.v0_1 import validate_and_freeze
FROZEN = validate_and_freeze(CONTROLLER, ROOT)
print('PASS:', FROZEN.run_id, FROZEN.route_id, FROZEN.config_sha256)


In [ ]:
# CELL 3 — frozen settings table
import pandas as pd
rows = []
for section, values in [('derived', FROZEN.derived), ('locked', FROZEN.locked)]:
    rows.extend({'section': section, 'setting': k, 'value': str(v)} for k, v in values.items())
pd.DataFrame(rows)


In [ ]:
# CELL 4 — frozen contract로 pipeline 실행
from urp4.hq.v0_1 import run_hq
STATUS = run_hq(CONTROLLER, ROOT)
STATUS


In [ ]:
# CELL 5 — input geometry manifest
import json
RUN_DIR = Path(STATUS['run_dir'])
json.loads((RUN_DIR / 'input_geometry_manifest.json').read_text(encoding='utf-8'))


In [ ]:
# CELL 6 — geometry QA
pd.DataFrame([STATUS['preflight']]).T.rename(columns={0: 'value'})


In [ ]:
# CELL 7 — descriptor 결과
from IPython.display import display
if STATUS.get('descriptor'):
    display(pd.read_csv(STATUS['descriptor']['csv']))
else:
    print('Descriptor execution was disabled or unavailable for this route.')


In [ ]:
# CELL 8 — slice QA / flagged review 상태
qa_rows = [
    {'gate': 'run', 'status': STATUS['status']},
    {'gate': 'route', 'status': STATUS['route_id']},
    {'gate': 'geometry_preflight', 'status': STATUS['preflight']['route_status']},
    {'gate': 'descriptor', 'status': (STATUS.get('descriptor') or {}).get('status', 'not_run')},
    {'gate': 'training', 'status': STATUS['training']['status']},
]
pd.DataFrame(qa_rows)


In [ ]:
# CELL 9 — export / output manifest
display(pd.read_csv(RUN_DIR / 'output_manifest.csv'))


In [ ]:
# CELL 10 — x-only quality/redundancy 준비 상태 (selection은 하지 않음)
path = RUN_DIR / 'x_only_readiness.json'
json.loads(path.read_text(encoding='utf-8')) if path.is_file() else {'status': 'descriptor_not_run'}


In [ ]:
# CELL 11 — feature-selection / training 상태와 필요 입력
json.loads((RUN_DIR / 'training_required_input_schema.json').read_text(encoding='utf-8'))


## Cell 12 — 실행 요약과 다음 단계 잠금

Feature selection과 Training은 아직 실행되지 않습니다. `target_y`가 비어 있거나 공식 modeling gate가 닫힌 상태에서는 HQ가 실패하도록 설계되어 있습니다. 구조인자 결과와 QA를 먼저 검토하세요.
